# Phase 11 — LangGraph fundamentals

This notebook is a focused LangGraph learning exercise. It builds and runs several tiny, deterministic state machines so that State, StateGraph, nodes, edges, START, END, state updates, conditional edges, and checkpointing can be observed directly.

The notebook intentionally contains no textbook retrieval, RAG context, embeddings, Qdrant, BM25, LLM call, tool call, agent, autonomous loop, or Agentic RAG behavior. Every node is ordinary deterministic Python code.


## 1. Core concepts

| Concept | What it does in this notebook |
| --- | --- |
| State | A typed dictionary carrying values between nodes. A reducer can define how repeated updates combine. |
| StateGraph | The builder that registers the state schema, nodes, and routes before compilation. |
| Node | A deterministic function that reads state and returns a partial state update. |
| Edge | A fixed transition that names the next node. |
| START | The graph entry marker. An edge from START selects the first node. |
| END | The graph exit marker. An edge to END stops the run. |
| Conditional edge | A route selected by a deterministic function of current state. |
| Checkpointer | A persistence layer that stores execution snapshots under a thread identifier. |


In [1]:
from __future__ import annotations

import json
import operator
from datetime import datetime, timezone
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import langgraph
from IPython.display import Markdown, display
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Business Knowledge AI project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_langgraph_basics_results.json"
STATUS_PATH = PROCESSED_DIR / "introduction_to_business_langgraph_basics_status.json"

print(f"LangGraph version: {langgraph.__version__ if hasattr(langgraph, '__version__') else 'installed'}")
print(f"Project root: {PROJECT_ROOT}")


LangGraph version: installed
Project root: /home/ubuntu/business-knowledge-ai


## 2. Example A — Fixed START → node_a → node_b → node_c → END graph

The linear example uses a State with a text value, a numerical counter, and a list of completed steps. The steps field uses an additive reducer, so each node appends its own label instead of replacing the earlier labels. Each node returns only the fields it changes; LangGraph merges those partial updates into the state.


In [2]:
class LinearState(TypedDict):
    message: str
    total: int
    steps: Annotated[list[str], operator.add]


def node_a(state: LinearState) -> dict:
    return {
        "message": state["message"] + " -> node_a",
        "total": state["total"] + 1,
        "steps": ["node_a"],
    }


def node_b(state: LinearState) -> dict:
    return {
        "message": state["message"] + " -> node_b",
        "total": state["total"] + 1,
        "steps": ["node_b"],
    }


def node_c(state: LinearState) -> dict:
    return {
        "message": state["message"] + " -> node_c",
        "total": state["total"] + 1,
        "steps": ["node_c"],
    }


linear_builder = StateGraph(LinearState)
linear_builder.add_node("node_a", node_a)
linear_builder.add_node("node_b", node_b)
linear_builder.add_node("node_c", node_c)
linear_builder.add_edge(START, "node_a")
linear_builder.add_edge("node_a", "node_b")
linear_builder.add_edge("node_b", "node_c")
linear_builder.add_edge("node_c", END)
linear_graph = linear_builder.compile()

linear_input = {"message": "START", "total": 0, "steps": []}
linear_result = linear_graph.invoke(linear_input)
assert linear_result == {
    "message": "START -> node_a -> node_b -> node_c",
    "total": 3,
    "steps": ["node_a", "node_b", "node_c"],
}

display(Markdown("### Actual linear-graph execution"))
print("Input:", linear_input)
print("Output:", linear_result)


### Actual linear-graph execution

Input: {'message': 'START', 'total': 0, 'steps': []}
Output: {'message': 'START -> node_a -> node_b -> node_c', 'total': 3, 'steps': ['node_a', 'node_b', 'node_c']}


## 3. Example B — Deterministic conditional edge

This graph starts at classify_number. Its route function is not an LLM decision: it computes parity with the modulo operator. Even inputs take the even_node path; odd inputs take the odd_node path. Both paths finish at END. Running both inputs makes the two allowed branches visible.


In [3]:
class RouteState(TypedDict):
    value: int
    route: str
    outcome: str
    steps: Annotated[list[str], operator.add]


def classify_number(state: RouteState) -> dict:
    route = "even_node" if state["value"] % 2 == 0 else "odd_node"
    return {"route": route, "steps": ["classify_number"]}


def choose_parity_path(state: RouteState) -> Literal["even_node", "odd_node"]:
    return state["route"]  # Deterministic selection from the state update above.


def even_node(state: RouteState) -> dict:
    return {
        "outcome": f"{state['value']} is even.",
        "steps": ["even_node"],
    }


def odd_node(state: RouteState) -> dict:
    return {
        "outcome": f"{state['value']} is odd.",
        "steps": ["odd_node"],
    }


conditional_builder = StateGraph(RouteState)
conditional_builder.add_node("classify_number", classify_number)
conditional_builder.add_node("even_node", even_node)
conditional_builder.add_node("odd_node", odd_node)
conditional_builder.add_edge(START, "classify_number")
conditional_builder.add_conditional_edges(
    "classify_number",
    choose_parity_path,
    {"even_node": "even_node", "odd_node": "odd_node"},
)
conditional_builder.add_edge("even_node", END)
conditional_builder.add_edge("odd_node", END)
conditional_graph = conditional_builder.compile()

conditional_results = {
    "even_input": conditional_graph.invoke(
        {"value": 8, "route": "", "outcome": "", "steps": []}
    ),
    "odd_input": conditional_graph.invoke(
        {"value": 7, "route": "", "outcome": "", "steps": []}
    ),
}
assert conditional_results["even_input"]["route"] == "even_node"
assert conditional_results["even_input"]["outcome"] == "8 is even."
assert conditional_results["odd_input"]["route"] == "odd_node"
assert conditional_results["odd_input"]["outcome"] == "7 is odd."

display(Markdown("### Actual conditional-graph executions"))
for label, result in conditional_results.items():
    print(f"{label}: {result}")


### Actual conditional-graph executions

even_input: {'value': 8, 'route': 'even_node', 'outcome': '8 is even.', 'steps': ['classify_number', 'even_node']}
odd_input: {'value': 7, 'route': 'odd_node', 'outcome': '7 is odd.', 'steps': ['classify_number', 'odd_node']}


## 4. Example C — Checkpointing and persistence

The next graph doubles a value once. It compiles with an in-memory MemorySaver checkpointer. A thread ID in the runtime configuration identifies the saved execution. After invocation, get_state retrieves the checkpoint snapshot and get_state_history exposes the recorded execution snapshots. This demonstrates persistence mechanics without a database, user memory, RAG history, or agent behavior.


In [4]:
class CheckpointState(TypedDict):
    value: int
    history: Annotated[list[str], operator.add]


def double_value(state: CheckpointState) -> dict:
    return {
        "value": state["value"] * 2,
        "history": [f"doubled {state['value']} to {state['value'] * 2}"],
    }


checkpointer = MemorySaver()
checkpoint_builder = StateGraph(CheckpointState)
checkpoint_builder.add_node("double_value", double_value)
checkpoint_builder.add_edge(START, "double_value")
checkpoint_builder.add_edge("double_value", END)
checkpoint_graph = checkpoint_builder.compile(checkpointer=checkpointer)

thread_config = {"configurable": {"thread_id": "phase11-checkpoint-demo"}}
checkpoint_input = {"value": 6, "history": []}
checkpoint_result = checkpoint_graph.invoke(checkpoint_input, config=thread_config)
saved_snapshot = checkpoint_graph.get_state(thread_config)
snapshot_history = list(checkpoint_graph.get_state_history(thread_config))

assert checkpoint_result == {"value": 12, "history": ["doubled 6 to 12"]}
assert saved_snapshot.values == checkpoint_result
assert len(snapshot_history) >= 2

display(Markdown("### Actual checkpointed execution"))
print("Thread ID:", thread_config["configurable"]["thread_id"])
print("Invocation result:", checkpoint_result)
print("Persisted state:", saved_snapshot.values)
print("Saved snapshot count:", len(snapshot_history))


### Actual checkpointed execution

Thread ID: phase11-checkpoint-demo
Invocation result: {'value': 12, 'history': ['doubled 6 to 12']}
Persisted state: {'value': 12, 'history': ['doubled 6 to 12']}
Saved snapshot count: 3


## 5. What the examples establish

The examples demonstrate deterministic graph orchestration at the smallest useful scale. State carries data, nodes return updates, ordinary edges form a fixed sequence, a conditional edge selects among named paths, and a checkpointer associates snapshots with a thread. The graphs do not infer routes, select tools, call a model, retrieve information, or loop autonomously.


In [5]:
results = {
    "phase": "11_langgraph_basics",
    "langgraph_version": getattr(langgraph, "__version__", "installed"),
    "scope": "Deterministic LangGraph fundamentals only; no RAG, agent, tool calling, or LLM.",
    "concepts_demonstrated": [
        "State",
        "StateGraph",
        "Nodes",
        "Edges",
        "START",
        "END",
        "State updates",
        "Conditional edges",
        "Checkpointing / persistence",
    ],
    "linear_example": {
        "topology": "START -> node_a -> node_b -> node_c -> END",
        "input": linear_input,
        "output": linear_result,
    },
    "conditional_example": {
        "topology": "START -> classify_number -> (even_node | odd_node) -> END",
        "outputs": conditional_results,
    },
    "checkpoint_example": {
        "thread_id": thread_config["configurable"]["thread_id"],
        "input": checkpoint_input,
        "output": checkpoint_result,
        "persisted_state": saved_snapshot.values,
        "snapshot_count": len(snapshot_history),
        "checkpointer": "MemorySaver",
    },
}
with RESULTS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(results, handle, indent=2, ensure_ascii=False)

status = {
    "phase": "11_langgraph_basics",
    "langgraph_installed": True,
    "langgraph_version": getattr(langgraph, "__version__", "installed"),
    "notebook_executed": True,
    "linear_graph_executed": True,
    "conditional_graph_executed": True,
    "checkpoint_graph_executed": True,
    "rag_implemented": False,
    "llm_invocation_implemented": False,
    "tool_calling_implemented": False,
    "agentic_control_flow_implemented": False,
    "results_artifact": str(RESULTS_PATH.relative_to(PROJECT_ROOT)),
    "status": "completed",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
}
with STATUS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(status, handle, indent=2, ensure_ascii=False)

print(json.dumps(status, indent=2))
print(f"Saved results: {RESULTS_PATH}")
print(f"Saved status: {STATUS_PATH}")


{
  "phase": "11_langgraph_basics",
  "langgraph_installed": true,
  "langgraph_version": "installed",
  "notebook_executed": true,
  "linear_graph_executed": true,
  "conditional_graph_executed": true,
  "checkpoint_graph_executed": true,
  "rag_implemented": false,
  "llm_invocation_implemented": false,
  "tool_calling_implemented": false,
  "agentic_control_flow_implemented": false,
  "results_artifact": "data/processed/introduction_to_business_langgraph_basics_results.json",
  "status": "completed",
  "executed_at_utc": "2026-08-14T13:34:59.830048+00:00"
}
Saved results: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_langgraph_basics_results.json
Saved status: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_langgraph_basics_status.json


## Phase boundary

Phase 11 is complete when these toy executions are understood and repeatable. It deliberately stops before combining LangGraph with retrieval or answer generation. A later phase can compose fixed retrieval and generation nodes into a deterministic RAG graph, but no such workflow is added here.
